In [1]:
import pandas as pd

# loading csv files
logon = pd.read_csv("../data/r4.2/logon.csv", nrows=300000)
device = pd.read_csv("../data/r4.2/device.csv", nrows=300000)
file = pd.read_csv("../data/r4.2/file.csv", nrows=300000)
http = pd.read_csv("../data/r4.2/http.csv", nrows=300000)
email = pd.read_csv("../data/r4.2/email.csv", nrows=300000)


# Convert date column
def process_time(df):
    df['date'] = pd.to_datetime(df['date'])
    df['day'] = df['date'].dt.date
    df['hour'] = df['date'].dt.hour
    return df


logon = process_time(logon)
file = process_time(file)
device = process_time(device)
http = process_time(http)
email = process_time(email)


# LOGIN FEATURES
login_features = logon.groupby(['user','day']).agg(
    first_login=('hour','min'),
    last_login=('hour','max'),
    night_activity=('hour', lambda x: ((x < 6) | (x > 20)).sum())
).reset_index()


# FILE FEATURES
file_features = file.groupby(['user','day']).agg(
    file_access=('filename','count')
).reset_index()


# USB FEATURES
device_features = device.groupby(['user','day']).agg(
    usb_usage=('activity', lambda x: (x == 'Connect').sum())
).reset_index()


# WEB FEATURES (Top 2 URLs in one column)
def top_urls(urls):
    top = urls.value_counts().head(2).index.tolist()

    while len(top) < 2:
        top.append("None")
    return " | ".join(top)

web_features = http.groupby(['user','day'])['url'].apply(top_urls).reset_index()

web_features = web_features.rename(columns={
    'url': 'top_urls'
})


# EXTERNAL EMAIL FEATURES
email_features = email.groupby(['user','day']).agg(
    external_emails=('to', lambda x: x.str.contains('@').sum())
).reset_index()


# MERGE ALL DATA
data = login_features

data = data.merge(file_features, on=['user','day'], how='left')
data = data.merge(device_features, on=['user','day'], how='left')
data = data.merge(web_features, on=['user','day'], how='left')
data = data.merge(email_features, on=['user','day'], how='left')

data = data.fillna(0)


# Rename columns
data = data.rename(columns={
    'user': 'user_id',
    'day': 'date'
})


# Risk Factor
data['risk_factor'] = (
    data['night_activity'] * 2 +
    data['file_access'] * 0.5 +
    data['usb_usage'] * 3 +
    data['external_emails'] * 1.5
)


# Threat Percentage
max_risk = data['risk_factor'].max()

data['threat_percentage'] = (data['risk_factor'] / max_risk) * 100


# Save processed dataset
data.to_csv("user_behavior_activity.csv", index=False)

print("Processed dataset saved.")

Processed dataset saved.
